In [1]:
%load_ext autoreload
%autoreload 2
import warnings
from pandas.errors import SettingWithCopyWarning

warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
from app.logger import *
import json5,json
import fitz #type: ignore

from app.utils import *
from app.konstant import get_config, get_regex
from pathlib import Path

from app.amc.fund_data import *
utils = Helper()

#PRE_DATA_SELECT IS REDUNDANT !!
# map and select are common dude so we gotta remove them !!

In [2]:
#INSURANCE FUND
amc_id = "37_0"
path = r"37_31-Aug-26_FS.pdf"
config = get_config("2026",amc_id)
regex = get_regex("2026")

object = Sundaram(config,regex,path)
title,path_pdf= object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [13]:
# object = Bandhan(config,regex,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [ ]:
title

In [14]:
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)

with open("data.json","w+") as file:
    json.dump(final_text,file)

save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
    
print(f"File Saved At: {save_path}")

File Saved At: c:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\working_fsparse\output\json\37_31-Aug-26_FS.json


In [37]:
pattern = "(?:Mr|Mrs?|Ms)\\.?\\s+([A-Za-z]+\\s+[A-Za-z]+).+?Total Experience\\:?\\s+(\\d{1,2}\\+?\\s+[Yy]ears?)\\s+Managing\\s+Since\\:?\\s+((?:Since\\s+)?[iI]nception|[A-Za-z]+\\s+\\d{1,2}\\,?\\s+\\d{4})"

for fund, content in final_text.items():
    # check = 'before.fund_manager'
    for key in content:
        if key.endswith(".fund_manager_details"):
            print(fund)
            text =re.sub("[^A-Za-z0-9\\s\\-\\(\\)\\.\\,\\+\\%\\:\\&]+", "",content[key]).strip()
            print(text)
            match = re.findall(pattern,text, re.IGNORECASE)
            print(match)

Parag Parikh Flexi Cap Fund
Name of the Fund Managers: Mr. Rajeev Thakkar: Chief Investment Officer - Equity and Director Total Experience: 32 Years Managing Since: Since Inception Mr. Raunak Onkar: Fund Manager Dedicated for Overseas Securities Total Experience: 17 Years Managing Since: Since Inception Mr. Raj Mehta: Executive Vice President and Fund Manager - Equity Total Experience: 13 Years Managing Since: September 1, 2025 Mr. Rukun Tarachandani - Executive Vice President & Fund Manager - Equity Total Experience: 13 Years Managing Since: May 16, 2022 Mr. Tejas Soman: Chief Investment Officer - Debt Total Experience: 12 Years Managing Since: September 1, 2025 Ms. Mansi Kariya: Associate Vice President & Fund Manager- Debt Total Experience: 11 Years Managing Since: December 22, 2023 Ms. Aishwarya Dhar: Senior Manager & Fund Manager- Debt Total Experience: 13 Years Managing Since: September 1, 2025
[('Rajeev Thakkar', '32 Years', 'Since Inception'), ('Raunak Onkar', '17 Years', 'Sinc

In [ ]:
import json, csv, unicodedata, os, ast, re
import pandas as pd
from pathlib import Path
from datetime import datetime

def sid_to_csv(json_path, output_folder=""):
    json_path = Path(json_path)

    output_folder = Path(output_folder) if output_folder else json_path.parent
    output_folder.mkdir(parents=True, exist_ok=True)

    csv_path = output_folder / f"{json_path.stem}.csv"

    
    with open(json_path, "r", encoding="utf-8") as f:
        doc = json.load(f)

    record = doc.get("value", {}) or {}

    def write_df(fh, df):
        df.to_csv(fh, index=False)
        fh.write("\n\n")  # 2 blank rows as section separator

    with open(csv_path, "w", encoding="utf-8", newline="") as fh:

        kv_rows = []
        for k in sorted(record.keys()):
            if k not in ("fund_manager", "load","field_location"):
                v = record.get(k)
                # print(v)
                if isinstance(v, (dict,list)):
                    v = " ".join(v)
                
                kv_rows.append({
                    "key": k,
                    "value": v if v is not None else ""
                })
            
            if k == "field_location":
                v = record.get(k)
                v = str(v)
                kv_rows.append(
                     "key": k,
                     "value": v if v is not None else ""
                )

        df1 = pd.DataFrame(kv_rows, columns=["key", "value"])
        write_df(fh, df1)

        fund_manager = record.get("fund_manager") or []
        if fund_manager:
            df2 = pd.DataFrame(fund_manager)
            write_df(fh, df2)

        load = record.get("load") or []
        if load:
            df3 = pd.DataFrame(load)
            write_df(fh, df3)

    return str(csv_path)

def csv_to_sid_json(csv_path, output_folder=""):
    csv_path = Path(csv_path)

    output_folder = Path(output_folder) if output_folder else csv_path.parent
    output_folder.mkdir(parents=True, exist_ok=True)

    json_path = output_folder / f"{csv_path.stem}.json"

    # ---------- split CSV into sections ----------
    sections = []
    current = []

    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in reader:
            if not any(cell.strip() for cell in row):
                if current:
                    sections.append(current)
                    current = []
            else:
                current.append(row)

        if current:
            sections.append(current)

    value = {}
    if sections:
        header, *rows = sections[0]
        key_idx = header.index("key")
        val_idx = header.index("value")

        for r in rows:
            k = r[key_idx].strip()
            v = r[val_idx].strip()

            if k == "field_location" and v:
                try:
                    v = ast.literal_eval(v)  # safe dict restore
                except Exception:
                    pass

            value[k] = v

    if len(sections) > 1:
        header, *rows = sections[1]
        fund_manager = []

        for r in rows:
            rec = {
                h: r[i].strip() if i < len(r) else ""
                for i, h in enumerate(header)
                if h
            }
            fund_manager.append(rec)

        value["fund_manager"] = fund_manager

    if len(sections) > 2:
        header, *rows = sections[2]
        load = []

        for r in rows:
            rec = {
                h: r[i].strip() if i < len(r) else ""
                for i, h in enumerate(header)
                if h
            }
            load.append(rec)

        value["load"] = load

    # ---------- METADATA ----------
    metadata = {
        "document_name": csv_path.name,
        "file_type": "sid",
        "process_date": datetime.today().strftime("%Y%m%d")
    }

    # ---------- FINAL JSON ----------
    output = {
        "metadata": metadata,
        "value": value
    }

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    return str(json_path)

def kim_to_csv(json_path, output_folder=""):
    json_path = Path(json_path)

    output_folder = Path(output_folder) if output_folder else json_path.parent
    output_folder.mkdir(parents=True, exist_ok=True)

    csv_path = output_folder / f"{json_path.stem}.csv"

    with open(json_path, "r", encoding="utf-8") as f:
        doc = json.load(f)

    record = (
        doc.get("records", [{}])[0]
        .get("value", {})
    ) or {}

    def write_df(fh, df):
        df.to_csv(fh, index=False)
        fh.write("\n\n")  # section separator

    with open(csv_path, "w", encoding="utf-8", newline="") as fh:

        # ================= DF1: Static KV =================
        kim_static_keys = [
            "amc_name",
            "main_scheme_name",
            "mutual_fund_name",
            "description",
            "field_location"   # ✅ added
        ]

        kv_rows = []
        for k in kim_static_keys:
            v = record.get(k, "")
            if isinstance(v, (dict, list)):
                v = str(v)
            kv_rows.append({
                "key": k,
                "value": v if v is not None else ""
            })

        df1 = pd.DataFrame(kv_rows, columns=["key", "value"])
        write_df(fh, df1)

        # ================= DF2: Asset Allocation =================
        allocation_rows = []

        for item in record.get("asset_allocation_pattern", []):
            row = {
                "instrument_type": item.get("instrument_type", ""),
                "risk_profile": item.get("risk_profile", "")
            }

            row.update({"min": "", "max": "", "total": ""})

            for alloc in item.get("allocation", []):
                alloc_type = alloc.get("type")
                if alloc_type in row:
                    row[alloc_type] = alloc.get("value", "")

            allocation_rows.append(row)

        if allocation_rows:
            df2 = pd.DataFrame(
                allocation_rows,
                columns=["instrument_type", "min", "max", "total", "risk_profile"]
            )
            write_df(fh, df2)

    return str(csv_path)

def csv_to_kim_json(csv_path, output_folder=""):
    csv_path = Path(csv_path)

    output_folder = Path(output_folder) if output_folder else csv_path.parent
    output_folder.mkdir(parents=True, exist_ok=True)

    json_path = output_folder / f"{csv_path.stem}.json"

    # ---------- split CSV into sections ----------
    sections, current = [], []

    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in reader:
            if not any(cell.strip() for cell in row):
                if current:
                    sections.append(current)
                    current = []
            else:
                current.append(row)
        if current:
            sections.append(current)

    value = {}
    if sections:
        header, *rows = sections[0]
        key_idx = header.index("key")
        val_idx = header.index("value")

        for r in rows:
            k = r[key_idx].strip()
            v = r[val_idx].strip()

            if k == "field_location" and v:
                try:
                    v = ast.literal_eval(v)  # ✅ safe restore
                except Exception:
                    v = []

            value[k] = v

    if len(sections) > 1:
        header, *rows = sections[1]
        asset_allocation_pattern = []

        for r in rows:
            row = {h: r[i].strip() if i < len(r) else "" for i, h in enumerate(header)}

            allocation = [
                {"type": "min", "value": row.get("min", "")},
                {"type": "max", "value": row.get("max", "")},
                {"type": "total", "value": row.get("total", "")},
            ]

            asset_allocation_pattern.append({
                "instrument_type": row.get("instrument_type", ""),
                "risk_profile": row.get("risk_profile", ""),
                "allocation": allocation
            })

        value["asset_allocation_pattern"] = asset_allocation_pattern

    # ================= METADATA =================
    metadata = {
        "document_name": csv_path.name,
        "file_type": "kim",
        "process_date": datetime.today().strftime("%Y%m%d")
    }

    output = {
        "metadata": metadata,
        "records": [{"value": value}]
    }
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    return json_path
    
path = r"C:\Users\kaustubh.keny\Downloads\100_18011_Jan-2026_1769580239_SID.json"
k_path = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\mywork-repo\notebook\fs_sidkim\27_18016_Jan-2026_1769757893_KIM.json"
str_path = sid_to_csv(path)
csv_to_sid_json(str_path)
# str_path = kim_to_csv(k_path)
# print(str_path)
# csv_to_kim_json(str_path)


In [56]:
def json_to_csv(json_path, output_dir="."):
    keys = {
            "static_keys": [
                "amc_name", "main_scheme_name", "mutual_fund_name", "benchmark_index",
                "monthly_aaum_date", "monthly_aaum_value", "scheme_launch_date",
                "min_addl_amt", "min_addl_amt_multiple", "min_amt", "min_amt_multiple"
            ],
            "load_keys": ["entry", "exit"],
            "metric_keys": [
                "alpha", "arithmetic_mean_ratio", "average_div_yield", "average_pb", "average_pe",
                "avg_maturity", "beta", "correlation_ratio", "downside_deviation", "information_ratio",
                "macaulay", "mod_duration", "port_turnover_ratio", "r_squared_ratio", "roe_ratio",
                "sharpe", "sortino_ratio", "std_dev", "tracking_error", "treynor_ratio",
                "upside_deviation", "ytm"
            ],
            "manager_keys": ["name", "managing_fund_since", "total_exp", "qualification"]
        }
    
    static_keys, load_keys, metric_keys, manager_keys = ( keys["static_keys"], keys["load_keys"], keys["metric_keys"], keys["manager_keys"] )

    with open(json_path, "r", encoding="utf-8") as f:
        doc = json.load(f)

    records = doc.get("records", [])
    max_manager = max((len(r["value"].get("fund_manager", [])) for r in records), default=1)

    # Build headers
    headers = static_keys + metric_keys
    for i in range(1, max_manager + 1):
        headers.extend([f"{k}_{i}" for k in manager_keys])
    headers.extend(load_keys)

    def flatten_to_row(value):
        row = []
        # static
        for k in static_keys:
            v = value.get(k, "")
            if isinstance(v, list):
                v = ", ".join(v)
            row.append(v)
        # metrics
        metric_map = {m.get("name"): m.get("value") for m in value.get("metrics", [])}
        row.extend([metric_map.get(k, "") for k in metric_keys])
        # fund managers
        managers = value.get("fund_manager", [])
        for i in range(max_manager):
            if i < len(managers):
                fm = managers[i]
                row.extend([fm.get(k, "") for k in manager_keys])
            else:
                row.extend([""] * len(manager_keys))
        
        # loads
        entry, exit_ = "", ""
        for l in value.get("load", []):
            if l.get("type") == "entry": entry = l.get("comment", "")
            elif l.get("type") == "exit": exit_ = l.get("comment", "")
        row.extend([entry, exit_])
        
        return row

    rows = [flatten_to_row(r["value"]) for r in records]

    # Timestamped filename
    base = os.path.splitext(os.path.basename(json_path))[0]
    csv_path = os.path.join(output_dir, f"{base}.csv")
    pd.DataFrame(rows, columns=headers).to_csv(csv_path, index=False, encoding="utf-8")
    return csv_path

def csv_to_fs_json(csv_path):
    df = pd.read_csv(csv_path, dtype=str, encoding="utf-8",encoding_errors="replace").fillna("")
    keys = {
            "field_location": "field_location",
            "static_keys": [
                "amc_name", "main_scheme_name", "mutual_fund_name", "benchmark_index",
                "monthly_aaum_date", "monthly_aaum_value", "scheme_launch_date",
                "min_addl_amt", "min_addl_amt_multiple", "min_amt", "min_amt_multiple"
            ],
            "load_keys": ["entry", "exit"],
            "metric_keys": [
                "alpha", "arithmetic_mean_ratio", "average_div_yield", "average_pb", "average_pe",
                "avg_maturity", "beta", "correlation_ratio", "downside_deviation", "information_ratio",
                "macaulay", "mod_duration", "port_turnover_ratio", "r_squared_ratio", "roe_ratio",
                "sharpe", "sortino_ratio", "std_dev", "tracking_error", "treynor_ratio",
                "upside_deviation", "ytm"
            ],
            "manager_keys": ["name", "managing_fund_since", "total_exp", "qualification"]
        }

    records = []
    for _, row in df.iterrows():
        value = {}
        for k in keys["static_keys"]:
            if k in row and row[k]:
                # if k == "benchmark_index":
                #     print(row[k])
                value[k] = row[k]

        loads = []
        if row.get("entry"):
            loads.append({"type": "entry", "comment": row["entry"]})
        if row.get("exit"):
            loads.append({"type": "exit", "comment": row["exit"]})
        if loads:
            value["load"] = loads

        metrics = []
        for k in keys["metric_keys"]:
            if row.get(k):
                metrics.append({"name": k, "value": row[k]})
        if metrics:
            value["metrics"] = metrics

        managers = []
        i = 1
        while f"name_{i}" in df.columns:
            fm = {}
            for k in keys["manager_keys"]:
                if row.get(f"{k}_{i}"):
                    fm[k] = row[f"{k}_{i}"]
            if fm:
                managers.append(fm)
            i += 1
        if managers:
            value["fund_manager"] = managers

        if row.get(keys["field_location"]):
            try:
                value["field_location"] = [json.loads(row[keys["field_location"]])]
            except Exception:
                pass

        records.append({"value": dict(sorted(value.items()))})

    csv_path = Path(csv_path)
    file_name = str(csv_path.name).replace(".csv",".pdf")
    return {
        "metadata": {
            "document_name": file_name,
            "file_type": "fs",
            "process_date": datetime.now().strftime("%Y%m%d"),
        },
        "records": records,
    }





# path = r"C:\Users\kaustubh.keny\Downloads\71_31-Dec-25_FS.json"
# json_to_csv(path)
path = r"C:\Users\kaustubh.keny\Downloads\38_31-Aug-26_FS.csv"
jsn = csv_to_fs_json(path)


utils.save_json(jsn, path.replace(".csv",".json"))